## Initializing Spark Session

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.config("spark.sql.session.timeZone", "UTC").appName("FinSentAnalysis").getOrCreate()

# Apple News and Price Data

In [0]:
WORKSPACE = "paid"
VOLUME = f'/Volumes/{WORKSPACE}/default/ensf612/'

### Apple Financial News Data

In [0]:
# global constants
API_KEY : str = ''
BEZINGA_URL : str = 'https://api.benzinga.com/api/v2/news'
STOCK_TICKER : str = 'AAPL'
HEADERS = {"accept": "application/json"}
PAGE_SIZE = 100
PAGE_LIMIT = 400

params = {
    'token': API_KEY,
    'displayOutput' : 'full',
    'pageSize' : PAGE_SIZE,
    "sort": "created:asc",
    'tickers': STOCK_TICKER,
    'channels' : 'news'
}

In [0]:
import requests

news_data = []

params['dateFrom'] = '2015-01-01'
params['dateTo'] = '2020-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()

params['dateFrom'] = '2021-01-01'
params['dateTo'] = '2023-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()

params['dateFrom'] = '2024-01-01'
params['dateTo'] = '2025-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()


In [0]:
import json

# Define the filename
dbfs_path = f"dbfs:{VOLUME}aapl_news.json"

dbutils.fs.put(dbfs_path, json.dumps(news_data), overwrite=True)

### Apple Stock Price Data

In [0]:
# global constants
API_KEY : str = ''
API_SECRET_KEY : str = ''
STOCK_TICKER : str = 'AAPL'
ALPACA_URL : str = f'https://data.alpaca.markets/v2/stocks/{STOCK_TICKER}/bars'
HEADERS = {
    "accept": "application/json",
    "APCA-API-KEY-ID": API_KEY,
    "APCA-API-SECRET-KEY": API_SECRET_KEY
    }

params = {
    'timeframe' : '1D',
    'limit' : 10000,
    'adjustment' : 'all'
}

In [0]:
import requests

stock_data = []

params['start'] = '2015-01-01'
params['end'] = '2018-12-31'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock1 = response.json()['bars']
stock_data += stock1

params['start'] = '2019-01-01'
params['end'] = '2021-12-31'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock2 = response.json()['bars']
stock_data += stock2

params['start'] = '2022-01-01'
params['end'] = '2025-11-05'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock3 = response.json()['bars']
stock_data += stock3

In [0]:
import json

# Define the filename
dbfs_path = f"dbfs:{VOLUME}aapl_price.json"

dbutils.fs.put(dbfs_path, json.dumps(stock_data), overwrite=True)